In [1]:
from embedder import Embedder

2026-06-27 10:20:10.259346955 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [2]:
text = "How does approximate nearest neighbor search work?"

v = Embedder().encode(text)
print(v[0])

-0.02058203437252893


In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [4]:
target_file = "02-vector-search/lessons/07-sqlitesearch-vector.md"

content = next(
    (doc["content"] for doc in documents if doc["filename"] == target_file),
    None
)

print(content)

# Vector Search with sqlitesearch

Video: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous section we used minsearch for vector search.

It works, but it has three problems:

- It rebuilds the index on every startup
- It keeps everything in memory
- It searches by brute force


With text search we never felt these. Indexing was fast because we
didn't embed anything. With vector search, indexing runs a neural
network over every document, so it takes a minute on our dataset.
Keeping everything in memory is fine here, but a larger dataset would
need too much space.

The third problem is brute-force search. For every query we compare the
query vector against every single document. With 1,000 documents this is
fine, probably even faster than anything smarter. But as the dataset
grows past 10,000 or so, it gets slow, and we'll want an approximate
method instead.

What we've done so far is exact nearest neighbor (NN) sea

In [5]:
q2_embed = Embedder().encode(content)
print(q2_embed[0])

-0.006691662831469935


In [6]:
v.dot(q2_embed)

np.float64(0.36107026789538205)

In [7]:
import numpy as np

from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)



In [8]:
embedder = Embedder()
contents = [chunk["content"] for chunk in chunks]
X = embedder.encode_batch(contents)
scores = X.dot(v)
print(f'highest-scoring chunk belong to (its filename)?: {chunks[np.argmax(scores)]["filename"]}')

highest-scoring chunk belong to (its filename)?: 02-vector-search/lessons/07-sqlitesearch-vector.md


In [9]:
from minsearch import VectorSearch, Index

vector_search = VectorSearch()
vector_search.fit(X, chunks)
vec = embedder.encode('What metric do we use to evaluate a search engine?')
print('File of First Result:', vector_search.search(vec, num_results=1)[0]["filename"])

File of First Result: 04-evaluation/lessons/05-search-metrics.md


In [10]:
text_index = Index(text_fields=["content"])
text_index.fit(chunks)
q5_query = 'How do I store vectors in PostgreSQL?'
vect_files = {doc["filename"] for doc in vector_search.search(embedder.encode(q5_query), num_results=5)}
text_files = {doc["filename"] for doc in text_index.search(q5_query, num_results=5)}
print('Unique to vector search:', vect_files - text_files)

Unique to vector search: {'02-vector-search/lessons/08-pgvector.md'}


In [11]:
def rrf(lists, k=60):
    scores, docs = {}, {}
    for l in lists:
        for rank, doc in enumerate(l):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    return [docs[k] for k in sorted(scores, key=scores.get, reverse=True)]

q6_v = vector_search.search(embedder.encode('How do I give the model access to tools?'), num_results=10)
q6_t = text_index.search('How do I give the model access to tools?', num_results=10)
print('Top Hybrid Ranked File:', rrf([q6_v, q6_t])[0]["filename"])

Top Hybrid Ranked File: 01-agentic-rag/lessons/13-function-calling.md
